# 02 — Data Preprocessing

**Project #8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**  
Network Data Analysis Laboratory 2025-2026, Politecnico di Milano

---

## Goals
1. Clean raw data (drop NaN, filter impossible values)
2. Encode categorical features (`traffic_type` → one-hot)
3. Standardise numeric features (StandardScaler)
4. Create reproducible train / val / test splits
5. Save processed arrays to `data/processed/` for downstream notebooks

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.loader import load_venue
from src.data.preprocessor import Preprocessor, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load raw data

In [ ]:
acc  = load_venue('acc_arena',  data_dir='../data/raw')
salt = load_venue('salt_tar',   data_dir='../data/raw')

print(f"ACC Arena  raw: {acc.shape}")
print(f"Salt & Tar raw: {salt.shape}")

## 2. Preprocessing pipeline — ACC Arena (main task)

In [ ]:
prep_acc = Preprocessor(
    val_size=0.10,
    test_size=0.15,
    random_state=RANDOM_SEED,
    remove_zero_throughput=True,
)

X_train, X_val, X_test, y_train, y_val, y_test = prep_acc.fit_transform(acc)

print(f"\nSplit sizes:")
print(f"  Train : X={X_train.shape}, y={y_train.shape}")
print(f"  Val   : X={X_val.shape},   y={y_val.shape}")
print(f"  Test  : X={X_test.shape},  y={y_test.shape}")
print(f"\nFeature names:")
print(prep_acc.feature_names_out_)

In [ ]:
# Verify scaling: train features should have ~0 mean, ~1 std
print('Feature mean (should be ~0 for numeric):')
print(np.round(X_train.mean(axis=0), 3))
print('Feature std  (should be ~1 for numeric):')
print(np.round(X_train.std(axis=0), 3))

In [ ]:
# Target distribution after cleaning
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, split_name, y in zip(axes, ['Train', 'Val', 'Test'], [y_train, y_val, y_test]):
    ax.hist(y, bins=60, color='steelblue', edgecolor='white')
    ax.set_title(f'{split_name} — Throughput (Mbps)')
    ax.set_xlabel('Throughput (Mbps)')
    ax.set_ylabel('Count')
plt.suptitle('Target Distribution per Split — ACC Arena')
plt.tight_layout()
plt.show()

## 3. Save ACC Arena processed arrays

In [ ]:
np.save(PROCESSED_DIR / 'acc_X_train.npy', X_train)
np.save(PROCESSED_DIR / 'acc_X_val.npy',   X_val)
np.save(PROCESSED_DIR / 'acc_X_test.npy',  X_test)
np.save(PROCESSED_DIR / 'acc_y_train.npy', y_train)
np.save(PROCESSED_DIR / 'acc_y_val.npy',   y_val)
np.save(PROCESSED_DIR / 'acc_y_test.npy',  y_test)

# Save preprocessor
prep_acc.save(PROCESSED_DIR / 'preprocessor_acc.pkl')
print('Saved ACC Arena arrays and preprocessor.')

## 4. Preprocessing pipeline — Salt & Tar (for Transfer Learning)

In [ ]:
prep_salt = Preprocessor(
    val_size=0.10,
    test_size=0.15,
    random_state=RANDOM_SEED,
)

X_salt_train, X_salt_val, X_salt_test, y_salt_train, y_salt_val, y_salt_test = (
    prep_salt.fit_transform(salt)
)

print(f"Salt & Tar splits:")
print(f"  Train : {X_salt_train.shape}")
print(f"  Val   : {X_salt_val.shape}")
print(f"  Test  : {X_salt_test.shape}")

In [ ]:
np.save(PROCESSED_DIR / 'salt_X_train.npy', X_salt_train)
np.save(PROCESSED_DIR / 'salt_X_val.npy',   X_salt_val)
np.save(PROCESSED_DIR / 'salt_X_test.npy',  X_salt_test)
np.save(PROCESSED_DIR / 'salt_y_train.npy', y_salt_train)
np.save(PROCESSED_DIR / 'salt_y_val.npy',   y_salt_val)
np.save(PROCESSED_DIR / 'salt_y_test.npy',  y_salt_test)

prep_salt.save(PROCESSED_DIR / 'preprocessor_salt.pkl')
print('Saved Salt & Tar arrays and preprocessor.')

## 5. Class balance check (traffic type)

In [ ]:
# Check one-hot encoding captured all traffic types
traffic_cols = [c for c in prep_acc.feature_names_out_ if c.startswith('traffic_type_')]
print('Traffic type OHE columns:', traffic_cols)

feat_idx = {name: i for i, name in enumerate(prep_acc.feature_names_out_)}
for col in traffic_cols:
    pct = X_train[:, feat_idx[col]].mean() * 100
    print(f"  {col}: {pct:.1f}% of train samples")

## 6. Summary

| Dataset     | Train   | Val    | Test   | Features |
|-------------|---------|--------|--------|----------|
| ACC Arena   | …       | …      | …      | …        |
| Salt & Tar  | …       | …      | …      | …        |